# 02 - Model Saving, Loading & Inference Pipeline

---

## Learning Objectives

By the end of this notebook, you will be able to:

- Save model checkpoints **with metadata** (hyperparameters, metrics, timestamp)
- Implement the **checkpoint pattern** to automatically save the best model during training
- Load a saved model and **resume training** or run inference
- Build a **complete inference pipeline**: load, preprocess, forward pass, postprocess
- Package inference logic into a reusable `predict()` function
- Write inference functions for **image classification** and **text classification**
- Use **batch inference** for efficient processing of multiple inputs

## Prerequisites

- PyTorch fundamentals (tensors, `nn.Module`, training loops)
- Basic understanding of model training (loss, optimizer, epochs)
- Familiarity with Python classes and file I/O
- Previous notebook: [01 - Experiment Tracking](01_Experiment_Tracking_Lightweight.ipynb)

## Table of Contents

1. [Why Save Models?](#1-why-save-models)
2. [Saving and Loading Basics](#2-saving-and-loading-basics)
3. [Checkpoint Pattern: Save Best During Training](#3-checkpoint-pattern-save-best-during-training)
4. [Saving with Rich Metadata](#4-saving-with-rich-metadata)
5. [Loading and Running Inference](#5-loading-and-running-inference)
6. [Complete Inference Pipeline](#6-complete-inference-pipeline)
7. [Image Classification Inference](#7-image-classification-inference)
8. [Text Classification Inference](#8-text-classification-inference)
9. [Batch Inference for Efficiency](#9-batch-inference-for-efficiency)
10. [Exercise: Build a Complete Inference Pipeline](#10-exercise-build-a-complete-inference-pipeline)
11. [Common Mistakes & Debugging Tips](#11-common-mistakes--debugging-tips)

In [ ]:
import sys
sys.path.insert(0, "../..")
from src.utils.seed import set_global_seed

import os
import json
import time
import shutil
import tempfile
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt

set_global_seed(42)

%matplotlib inline
plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['figure.dpi'] = 100

# We will save all checkpoints to a temp directory for this notebook
CHECKPOINT_DIR = Path(tempfile.mkdtemp(prefix="dl800_nb02_"))
print(f"Checkpoint directory: {CHECKPOINT_DIR}")

---
## 1. Why Save Models?

Training a deep learning model can take minutes, hours, or even days. Without saving:

- **Wasted compute**: if training crashes at epoch 95/100, you lose everything
- **No deployment**: you cannot serve a model that only lives in RAM
- **No reproducibility**: you cannot re-examine results from last week

### What to save

| Component | Why |
|---|---|
| `model.state_dict()` | Learned weights and biases |
| `optimizer.state_dict()` | Momentum buffers, learning rate state (for resuming) |
| Hyperparameters | Rebuild the model architecture identically |
| Metrics | Know which checkpoint is best |
| Epoch number | Resume training from the correct point |
| Timestamp | Audit trail |

---
## 2. Saving and Loading Basics

PyTorch provides two approaches:

| Approach | Pros | Cons |
|---|---|---|
| `torch.save(model.state_dict(), path)` | Portable, flexible, recommended | Must recreate model class first |
| `torch.save(model, path)` | One-line save | Brittle -- breaks if code changes |

**Best practice:** Always save `state_dict()`, never the full model object.

In [ ]:
# Define a simple model for demonstration
class SimpleMLP(nn.Module):
    """Multi-layer perceptron for classification."""
    def __init__(self, input_dim: int, hidden_dim: int, output_dim: int,
                 dropout: float = 0.0):
        super().__init__()
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.output_dim = output_dim
        self.dropout = dropout
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, output_dim),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


# Create and save
model = SimpleMLP(input_dim=10, hidden_dim=64, output_dim=3)
save_path = CHECKPOINT_DIR / "basic_model.pt"

# Save state_dict (recommended)
torch.save(model.state_dict(), save_path)
print(f"Saved state_dict to {save_path}")
print(f"File size: {save_path.stat().st_size / 1024:.1f} KB")

In [ ]:
# Load state_dict into a new model instance
loaded_model = SimpleMLP(input_dim=10, hidden_dim=64, output_dim=3)
loaded_model.load_state_dict(torch.load(save_path, weights_only=True))
loaded_model.eval()  # Always set to eval mode for inference

# Verify: outputs should be identical
test_input = torch.randn(1, 10)
with torch.no_grad():
    original_out = model(test_input)
    loaded_out = loaded_model(test_input)

print(f"Outputs match: {torch.allclose(original_out, loaded_out)}")
print(f"Original output: {original_out.squeeze().tolist()}")
print(f"Loaded output:   {loaded_out.squeeze().tolist()}")

---
## 3. Checkpoint Pattern: Save Best During Training

The **checkpoint pattern** saves the model whenever validation loss improves. This ensures you never lose the best model, even if training later overfits or crashes.

```
for epoch in range(epochs):
    train(...)
    val_loss = validate(...)
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        save_checkpoint(model, optimizer, epoch, val_loss)
```

In [ ]:
# Create synthetic dataset
set_global_seed(42)

N_SAMPLES = 1000
N_FEATURES = 10
N_CLASSES = 3

X = torch.randn(N_SAMPLES, N_FEATURES)
true_w = torch.randn(N_FEATURES, N_CLASSES)
y = (X @ true_w).argmax(dim=1)

split = int(0.8 * N_SAMPLES)
X_train, X_val = X[:split], X[split:]
y_train, y_val = y[:split], y[split:]

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=64)

print(f"Train: {X_train.shape[0]}, Val: {X_val.shape[0]}, Features: {N_FEATURES}, Classes: {N_CLASSES}")

In [ ]:
def save_checkpoint(model: nn.Module, optimizer: torch.optim.Optimizer,
                    epoch: int, val_loss: float, path: str) -> None:
    """Save a training checkpoint.

    Args:
        model: The model to save.
        optimizer: Optimizer (for resuming training).
        epoch: Current epoch number.
        val_loss: Validation loss at this epoch.
        path: File path to save the checkpoint.
    """
    checkpoint = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "val_loss": val_loss,
    }
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    torch.save(checkpoint, path)


def load_checkpoint(path: str, model: nn.Module,
                    optimizer: Optional[torch.optim.Optimizer] = None
                    ) -> Dict[str, Any]:
    """Load a checkpoint. Restores model (and optionally optimizer) state.

    Returns:
        The full checkpoint dict (contains epoch, val_loss, etc.).
    """
    checkpoint = torch.load(path, weights_only=False)
    model.load_state_dict(checkpoint["model_state_dict"])
    if optimizer is not None and "optimizer_state_dict" in checkpoint:
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    return checkpoint

In [ ]:
# Training loop with checkpoint pattern
set_global_seed(42)

model = SimpleMLP(input_dim=N_FEATURES, hidden_dim=64, output_dim=N_CLASSES)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
loss_fn = nn.CrossEntropyLoss()

best_val_loss = float("inf")
best_epoch = -1
checkpoint_path = str(CHECKPOINT_DIR / "best_model.pt")
history = {"train_loss": [], "val_loss": []}

EPOCHS = 30

for epoch in range(1, EPOCHS + 1):
    # --- Train ---
    model.train()
    epoch_loss = 0.0
    n_batches = 0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        out = model(X_batch)
        loss = loss_fn(out, y_batch)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        n_batches += 1
    train_loss = epoch_loss / n_batches

    # --- Validate ---
    model.eval()
    val_loss_sum = 0.0
    n_val = 0
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            out = model(X_batch)
            val_loss_sum += loss_fn(out, y_batch).item()
            n_val += 1
    val_loss = val_loss_sum / n_val

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)

    # --- Checkpoint if improved ---
    saved_flag = ""
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch
        save_checkpoint(model, optimizer, epoch, val_loss, checkpoint_path)
        saved_flag = " <-- saved"

    if epoch % 5 == 0 or saved_flag:
        print(f"Epoch {epoch:3d}/{EPOCHS} | Train Loss: {train_loss:.4f} "
              f"| Val Loss: {val_loss:.4f}{saved_flag}")

print(f"\nBest model saved at epoch {best_epoch} with val_loss={best_val_loss:.4f}")

In [ ]:
# Plot training curves with best checkpoint marked
fig, ax = plt.subplots()
epochs_range = range(1, len(history["train_loss"]) + 1)
ax.plot(epochs_range, history["train_loss"], label="Train Loss", marker="o", markersize=3)
ax.plot(epochs_range, history["val_loss"], label="Val Loss", marker="s", markersize=3)
ax.axvline(x=best_epoch, color="red", linestyle="--", alpha=0.7, label=f"Best checkpoint (epoch {best_epoch})")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title("Training with Checkpoint Pattern")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 4. Saving with Rich Metadata

A production checkpoint should contain everything needed to **recreate the exact same model**. This includes hyperparameters, training metrics, and environment info.

In [ ]:
def save_checkpoint_with_metadata(
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    epoch: int,
    metrics: Dict[str, float],
    hyperparams: Dict[str, Any],
    path: str,
) -> None:
    """Save a checkpoint with full metadata for reproducibility.

    Args:
        model: The trained model.
        optimizer: The optimizer.
        epoch: Current epoch.
        metrics: Dict of metric names to values (e.g., val_loss, val_acc).
        hyperparams: Dict of hyperparameters used for training.
        path: File path to save.
    """
    checkpoint = {
        # Model and optimizer state
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        # Training progress
        "epoch": epoch,
        "metrics": metrics,
        # Hyperparameters (needed to rebuild model)
        "hyperparams": hyperparams,
        # Metadata
        "timestamp": datetime.now().isoformat(),
        "pytorch_version": torch.__version__,
        "model_class": model.__class__.__name__,
    }
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    torch.save(checkpoint, path)
    print(f"Checkpoint saved: epoch={epoch}, metrics={metrics}")


# Save with metadata
hyperparams = {
    "input_dim": N_FEATURES,
    "hidden_dim": 64,
    "output_dim": N_CLASSES,
    "dropout": 0.0,
    "lr": 0.01,
    "batch_size": 32,
    "optimizer": "Adam",
}

rich_path = str(CHECKPOINT_DIR / "best_model_rich.pt")
save_checkpoint_with_metadata(
    model=model,
    optimizer=optimizer,
    epoch=best_epoch,
    metrics={"val_loss": best_val_loss, "train_loss": history["train_loss"][best_epoch - 1]},
    hyperparams=hyperparams,
    path=rich_path,
)

In [ ]:
# Inspect the saved checkpoint contents
checkpoint = torch.load(rich_path, weights_only=False)

print("Checkpoint keys:")
for key in checkpoint:
    if key.endswith("_state_dict"):
        print(f"  {key}: <state dict with {len(checkpoint[key])} entries>")
    else:
        print(f"  {key}: {checkpoint[key]}")

---
## 5. Loading and Running Inference

To load a checkpoint for inference:

1. Read hyperparameters from the checkpoint
2. Reconstruct the model architecture
3. Load the saved weights
4. Set the model to `eval()` mode
5. Run forward pass with `torch.no_grad()`

In [ ]:
def load_model_for_inference(checkpoint_path: str) -> Tuple[nn.Module, Dict]:
    """Load a model from a rich checkpoint for inference.

    Returns:
        Tuple of (model in eval mode, checkpoint metadata dict).
    """
    checkpoint = torch.load(checkpoint_path, weights_only=False)
    hp = checkpoint["hyperparams"]

    # Reconstruct model from hyperparameters
    model = SimpleMLP(
        input_dim=hp["input_dim"],
        hidden_dim=hp["hidden_dim"],
        output_dim=hp["output_dim"],
        dropout=hp.get("dropout", 0.0),
    )
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    metadata = {
        "epoch": checkpoint["epoch"],
        "metrics": checkpoint["metrics"],
        "timestamp": checkpoint["timestamp"],
        "hyperparams": hp,
    }
    return model, metadata


# Load and run inference
inference_model, meta = load_model_for_inference(rich_path)
print(f"Loaded model from epoch {meta['epoch']}")
print(f"  Metrics: {meta['metrics']}")
print(f"  Saved at: {meta['timestamp']}")

# Single sample inference
sample = torch.randn(1, N_FEATURES)
with torch.no_grad():
    logits = inference_model(sample)
    probs = F.softmax(logits, dim=1)
    predicted_class = probs.argmax(dim=1).item()

print(f"\nSample inference:")
print(f"  Logits: {logits.squeeze().tolist()}")
print(f"  Probabilities: {[f'{p:.4f}' for p in probs.squeeze().tolist()]}")
print(f"  Predicted class: {predicted_class}")

---
## 6. Complete Inference Pipeline

A production inference pipeline has three stages:

```
raw input --> preprocess --> model.forward() --> postprocess --> output
```

Wrapping this in a single `predict()` function makes it easy to use.

In [ ]:
class InferencePipeline:
    """Complete inference pipeline: load, preprocess, predict, postprocess.

    Args:
        checkpoint_path: Path to a rich checkpoint file.
        class_names: Optional list of human-readable class names.
        device: Device to run inference on.
    """

    def __init__(self, checkpoint_path: str,
                 class_names: Optional[List[str]] = None,
                 device: str = "cpu"):
        self.device = torch.device(device)
        self.model, self.metadata = load_model_for_inference(checkpoint_path)
        self.model = self.model.to(self.device)
        self.class_names = class_names

    def preprocess(self, raw_input: np.ndarray) -> torch.Tensor:
        """Convert raw numpy input to a model-ready tensor."""
        if isinstance(raw_input, np.ndarray):
            tensor = torch.from_numpy(raw_input).float()
        else:
            tensor = torch.tensor(raw_input, dtype=torch.float32)
        # Ensure batch dimension
        if tensor.ndim == 1:
            tensor = tensor.unsqueeze(0)
        return tensor.to(self.device)

    def postprocess(self, logits: torch.Tensor) -> List[Dict[str, Any]]:
        """Convert raw logits to human-readable predictions."""
        probs = F.softmax(logits, dim=1)
        top_probs, top_indices = probs.topk(k=min(3, probs.shape[1]), dim=1)

        results = []
        for i in range(logits.shape[0]):
            pred_class = top_indices[i, 0].item()
            pred_label = (self.class_names[pred_class]
                          if self.class_names else str(pred_class))
            result = {
                "predicted_class": pred_class,
                "predicted_label": pred_label,
                "confidence": top_probs[i, 0].item(),
                "top_k": [
                    {
                        "class": top_indices[i, j].item(),
                        "label": (self.class_names[top_indices[i, j].item()]
                                  if self.class_names else str(top_indices[i, j].item())),
                        "probability": top_probs[i, j].item(),
                    }
                    for j in range(top_probs.shape[1])
                ],
            }
            results.append(result)
        return results

    @torch.no_grad()
    def predict(self, raw_input) -> List[Dict[str, Any]]:
        """End-to-end prediction: preprocess -> forward -> postprocess."""
        tensor = self.preprocess(raw_input)
        logits = self.model(tensor)
        return self.postprocess(logits)


# Demonstrate
pipeline = InferencePipeline(
    checkpoint_path=rich_path,
    class_names=["Class A", "Class B", "Class C"],
)

# Single prediction
sample_np = np.random.randn(N_FEATURES).astype(np.float32)
results = pipeline.predict(sample_np)
print("Single prediction:")
print(f"  Label: {results[0]['predicted_label']}")
print(f"  Confidence: {results[0]['confidence']:.4f}")
print(f"  Top-3:")
for entry in results[0]["top_k"]:
    print(f"    {entry['label']}: {entry['probability']:.4f}")

In [ ]:
# Batch prediction
batch_np = np.random.randn(5, N_FEATURES).astype(np.float32)
batch_results = pipeline.predict(batch_np)

print("Batch predictions (5 samples):")
for i, res in enumerate(batch_results):
    print(f"  Sample {i}: {res['predicted_label']} (conf={res['confidence']:.4f})")

---
## 7. Image Classification Inference

For image classification, the preprocessing step includes:
- Resizing to the expected input dimensions
- Converting to tensor
- Normalizing with dataset statistics

We simulate this with sklearn digits (8x8 grayscale images) to avoid large downloads.

In [ ]:
from sklearn.datasets import load_digits

# Load small image dataset
digits = load_digits()
X_digits = torch.tensor(digits.data, dtype=torch.float32)
y_digits = torch.tensor(digits.target, dtype=torch.long)

# Normalize to [0, 1]
X_digits = X_digits / 16.0

# Train/test split
set_global_seed(42)
perm = torch.randperm(len(X_digits))
split = int(0.8 * len(X_digits))
X_img_train = X_digits[perm[:split]]
y_img_train = y_digits[perm[:split]]
X_img_test = X_digits[perm[split:]]
y_img_test = y_digits[perm[split:]]

print(f"Image features: {X_img_train.shape[1]} (8x8 flattened)")
print(f"Train: {X_img_train.shape[0]}, Test: {X_img_test.shape[0]}, Classes: 10")

In [ ]:
# Train a digit classifier
set_global_seed(42)

img_model = SimpleMLP(input_dim=64, hidden_dim=128, output_dim=10, dropout=0.1)
img_optimizer = torch.optim.Adam(img_model.parameters(), lr=0.001)
img_loss_fn = nn.CrossEntropyLoss()

img_train_loader = DataLoader(TensorDataset(X_img_train, y_img_train),
                              batch_size=64, shuffle=True)

# Quick training
best_img_loss = float("inf")
img_ckpt_path = str(CHECKPOINT_DIR / "digit_classifier.pt")

for epoch in range(1, 31):
    img_model.train()
    for Xb, yb in img_train_loader:
        img_optimizer.zero_grad()
        loss = img_loss_fn(img_model(Xb), yb)
        loss.backward()
        img_optimizer.step()

    # Evaluate on test set
    img_model.eval()
    with torch.no_grad():
        test_logits = img_model(X_img_test)
        test_loss = img_loss_fn(test_logits, y_img_test).item()
        test_acc = (test_logits.argmax(1) == y_img_test).float().mean().item()

    if test_loss < best_img_loss:
        best_img_loss = test_loss
        save_checkpoint_with_metadata(
            img_model, img_optimizer, epoch,
            metrics={"test_loss": test_loss, "test_acc": test_acc},
            hyperparams={"input_dim": 64, "hidden_dim": 128, "output_dim": 10,
                         "dropout": 0.1, "lr": 0.001},
            path=img_ckpt_path,
        )

print(f"\nFinal test accuracy: {test_acc:.4f}")

In [ ]:
def predict_digit(image_flat: np.ndarray, checkpoint_path: str) -> Dict[str, Any]:
    """Inference function for digit classification.

    Args:
        image_flat: Flattened 8x8 image as numpy array (64 values in [0, 16]).
        checkpoint_path: Path to saved checkpoint.

    Returns:
        Dict with predicted digit, confidence, and top-3 predictions.
    """
    # 1. Preprocess: normalize and convert to tensor
    image_normalized = image_flat.astype(np.float32) / 16.0
    tensor = torch.from_numpy(image_normalized).unsqueeze(0)  # Add batch dim

    # 2. Load model
    model, meta = load_model_for_inference(checkpoint_path)

    # 3. Forward pass
    with torch.no_grad():
        logits = model(tensor)
        probs = F.softmax(logits, dim=1).squeeze()

    # 4. Postprocess
    top3_probs, top3_idx = probs.topk(3)
    return {
        "predicted_digit": top3_idx[0].item(),
        "confidence": top3_probs[0].item(),
        "top_3": [(idx.item(), prob.item()) for idx, prob in zip(top3_idx, top3_probs)],
    }


# Test on a sample
sample_idx = 0
sample_image = digits.data[perm[split + sample_idx]]  # Raw [0, 16] range
true_label = digits.target[perm[split + sample_idx]]

result = predict_digit(sample_image, img_ckpt_path)
print(f"True digit: {true_label}")
print(f"Predicted: {result['predicted_digit']} (confidence: {result['confidence']:.4f})")
print(f"Top-3: {result['top_3']}")

# Visualize
fig, ax = plt.subplots(1, 1, figsize=(3, 3))
ax.imshow(sample_image.reshape(8, 8), cmap="gray")
ax.set_title(f"True: {true_label}, Predicted: {result['predicted_digit']}")
ax.axis("off")
plt.tight_layout()
plt.show()

---
## 8. Text Classification Inference

For text classification, preprocessing converts raw strings into integer sequences. We build a small sentiment classifier on synthetic data.

In [ ]:
# Create a simple synthetic sentiment dataset
set_global_seed(42)

positive_words = ["good", "great", "excellent", "amazing", "wonderful",
                  "fantastic", "love", "best", "happy", "perfect"]
negative_words = ["bad", "terrible", "awful", "worst", "hate",
                  "horrible", "poor", "boring", "ugly", "disaster"]
neutral_words = ["the", "a", "is", "it", "this", "movie", "film", "was", "very", "really"]

texts = []
labels = []
rng = np.random.RandomState(42)

for _ in range(200):
    # Positive samples
    n_words = rng.randint(3, 8)
    words = list(rng.choice(neutral_words, n_words // 2))
    words += list(rng.choice(positive_words, n_words - n_words // 2))
    rng.shuffle(words)
    texts.append(" ".join(words))
    labels.append(1)

    # Negative samples
    n_words = rng.randint(3, 8)
    words = list(rng.choice(neutral_words, n_words // 2))
    words += list(rng.choice(negative_words, n_words - n_words // 2))
    rng.shuffle(words)
    texts.append(" ".join(words))
    labels.append(0)

print(f"Dataset size: {len(texts)}")
print(f"Sample positive: '{texts[0]}' -> {labels[0]}")
print(f"Sample negative: '{texts[1]}' -> {labels[1]}")

In [ ]:
from src.utils.text_preprocessing import build_vocab, texts_to_sequences

# Build vocabulary and convert to sequences
vocab = build_vocab(texts, special_tokens=["<PAD>", "<UNK>"])
MAX_LEN = 10
sequences = texts_to_sequences(texts, vocab, max_len=MAX_LEN)

X_text = torch.tensor(sequences, dtype=torch.long)
y_text = torch.tensor(labels, dtype=torch.long)

# Split
split_text = int(0.8 * len(X_text))
X_text_train, X_text_test = X_text[:split_text], X_text[split_text:]
y_text_train, y_text_test = y_text[:split_text], y_text[split_text:]

print(f"Vocab size: {len(vocab)}")
print(f"Sequence shape: {X_text.shape}")
print(f"Sample sequence: {X_text[0].tolist()}")

In [ ]:
class TextClassifier(nn.Module):
    """Simple embedding + mean pooling + linear classifier."""

    def __init__(self, vocab_size: int, embed_dim: int, hidden_dim: int,
                 output_dim: int, pad_idx: int = 0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.fc = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x shape: (batch, seq_len)
        embedded = self.embedding(x)  # (batch, seq_len, embed_dim)
        # Mean pooling (ignore padding)
        mask = (x != 0).unsqueeze(-1).float()  # (batch, seq_len, 1)
        pooled = (embedded * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
        return self.fc(pooled)


# Train
set_global_seed(42)

text_hp = {
    "vocab_size": len(vocab),
    "embed_dim": 32,
    "hidden_dim": 32,
    "output_dim": 2,
    "pad_idx": 0,
    "lr": 0.01,
    "max_len": MAX_LEN,
}

text_model = TextClassifier(
    vocab_size=text_hp["vocab_size"],
    embed_dim=text_hp["embed_dim"],
    hidden_dim=text_hp["hidden_dim"],
    output_dim=text_hp["output_dim"],
    pad_idx=text_hp["pad_idx"],
)
text_optimizer = torch.optim.Adam(text_model.parameters(), lr=text_hp["lr"])
text_loss_fn = nn.CrossEntropyLoss()

text_train_loader = DataLoader(
    TensorDataset(X_text_train, y_text_train), batch_size=32, shuffle=True
)

text_ckpt_path = str(CHECKPOINT_DIR / "text_classifier.pt")
best_text_loss = float("inf")

for epoch in range(1, 21):
    text_model.train()
    for Xb, yb in text_train_loader:
        text_optimizer.zero_grad()
        loss = text_loss_fn(text_model(Xb), yb)
        loss.backward()
        text_optimizer.step()

    # Evaluate
    text_model.eval()
    with torch.no_grad():
        test_logits = text_model(X_text_test)
        test_loss = text_loss_fn(test_logits, y_text_test).item()
        test_acc = (test_logits.argmax(1) == y_text_test).float().mean().item()

    if test_loss < best_text_loss:
        best_text_loss = test_loss
        # Save vocab alongside checkpoint
        text_hp_save = {**text_hp}
        torch.save({
            "model_state_dict": text_model.state_dict(),
            "hyperparams": text_hp_save,
            "vocab": vocab,
            "epoch": epoch,
            "metrics": {"test_loss": test_loss, "test_acc": test_acc},
            "timestamp": datetime.now().isoformat(),
            "class_names": ["negative", "positive"],
        }, text_ckpt_path)

    if epoch % 5 == 0:
        print(f"Epoch {epoch:2d} | Loss: {test_loss:.4f} | Acc: {test_acc:.4f}")

print(f"\nBest test loss: {best_text_loss:.4f}")

In [ ]:
def predict_sentiment(text: str, checkpoint_path: str) -> Dict[str, Any]:
    """Inference function for text sentiment classification.

    Args:
        text: Raw input string.
        checkpoint_path: Path to saved text classifier checkpoint.

    Returns:
        Dict with predicted sentiment, confidence, and probabilities.
    """
    # 1. Load checkpoint (includes vocab and hyperparams)
    ckpt = torch.load(checkpoint_path, weights_only=False)
    hp = ckpt["hyperparams"]
    saved_vocab = ckpt["vocab"]
    class_names = ckpt.get("class_names", ["negative", "positive"])

    # 2. Reconstruct and load model
    model = TextClassifier(
        vocab_size=hp["vocab_size"],
        embed_dim=hp["embed_dim"],
        hidden_dim=hp["hidden_dim"],
        output_dim=hp["output_dim"],
        pad_idx=hp["pad_idx"],
    )
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()

    # 3. Preprocess: tokenize and convert to sequence
    sequence = texts_to_sequences([text], saved_vocab, max_len=hp["max_len"])
    tensor = torch.tensor(sequence, dtype=torch.long)

    # 4. Forward pass
    with torch.no_grad():
        logits = model(tensor)
        probs = F.softmax(logits, dim=1).squeeze()

    # 5. Postprocess
    predicted_idx = probs.argmax().item()
    return {
        "text": text,
        "predicted_sentiment": class_names[predicted_idx],
        "confidence": probs[predicted_idx].item(),
        "probabilities": {name: probs[i].item() for i, name in enumerate(class_names)},
    }


# Test
test_sentences = [
    "this movie is great and amazing",
    "terrible film really boring and awful",
    "it was a movie",
]

for sent in test_sentences:
    result = predict_sentiment(sent, text_ckpt_path)
    print(f"'{result['text']}'")
    print(f"  -> {result['predicted_sentiment']} (conf={result['confidence']:.4f})")
    print(f"     probs: {result['probabilities']}")
    print()

---
## 9. Batch Inference for Efficiency

Processing one sample at a time wastes GPU parallelism. Batch inference is significantly faster.

**Key idea**: collect multiple inputs, stack them into a single tensor, run one forward pass.

In [ ]:
def batch_predict_sentiment(
    texts_list: List[str],
    checkpoint_path: str,
    batch_size: int = 32,
) -> List[Dict[str, Any]]:
    """Batch inference for text sentiment classification.

    Args:
        texts_list: List of raw text strings.
        checkpoint_path: Path to checkpoint.
        batch_size: Number of samples per forward pass.

    Returns:
        List of prediction dicts, one per input text.
    """
    # Load model once
    ckpt = torch.load(checkpoint_path, weights_only=False)
    hp = ckpt["hyperparams"]
    saved_vocab = ckpt["vocab"]
    class_names = ckpt.get("class_names", ["negative", "positive"])

    model = TextClassifier(
        vocab_size=hp["vocab_size"],
        embed_dim=hp["embed_dim"],
        hidden_dim=hp["hidden_dim"],
        output_dim=hp["output_dim"],
        pad_idx=hp["pad_idx"],
    )
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()

    # Convert all texts to sequences
    all_sequences = texts_to_sequences(texts_list, saved_vocab, max_len=hp["max_len"])
    all_tensors = torch.tensor(all_sequences, dtype=torch.long)

    # Process in batches
    all_results = []
    with torch.no_grad():
        for start in range(0, len(texts_list), batch_size):
            end = min(start + batch_size, len(texts_list))
            batch_tensor = all_tensors[start:end]
            logits = model(batch_tensor)
            probs = F.softmax(logits, dim=1)

            for i in range(probs.shape[0]):
                pred_idx = probs[i].argmax().item()
                all_results.append({
                    "text": texts_list[start + i],
                    "predicted_sentiment": class_names[pred_idx],
                    "confidence": probs[i, pred_idx].item(),
                })

    return all_results


# Benchmark: single vs batch
many_texts = texts[:100]

# Single inference
t0 = time.perf_counter()
single_results = [predict_sentiment(t, text_ckpt_path) for t in many_texts]
single_time = time.perf_counter() - t0

# Batch inference
t0 = time.perf_counter()
batch_results = batch_predict_sentiment(many_texts, text_ckpt_path, batch_size=32)
batch_time = time.perf_counter() - t0

print(f"Single inference ({len(many_texts)} samples): {single_time:.3f}s")
print(f"Batch inference  ({len(many_texts)} samples): {batch_time:.3f}s")
print(f"Speedup: {single_time / batch_time:.1f}x")

# Verify results match
match_count = sum(
    1 for s, b in zip(single_results, batch_results)
    if s["predicted_sentiment"] == b["predicted_sentiment"]
)
print(f"Predictions match: {match_count}/{len(many_texts)}")

---
## 10. Exercise: Build a Complete Inference Pipeline

**Task:** Build a complete, self-contained inference pipeline for the digit classifier.

Requirements:
1. Write a `DigitInferencePipeline` class with:
   - `__init__`: load checkpoint, rebuild model, extract metadata
   - `preprocess`: accept raw 8x8 numpy array, normalize, convert to tensor
   - `predict_single`: classify one image, return label + confidence
   - `predict_batch`: classify a batch of images efficiently
2. Test on 10 random samples from the test set
3. Print accuracy and average confidence

```python
class DigitInferencePipeline:
    def __init__(self, checkpoint_path: str):
        # TODO: load checkpoint, rebuild model, set eval mode
        pass

    def preprocess(self, images: np.ndarray) -> torch.Tensor:
        # TODO: normalize [0,16] -> [0,1], convert to float tensor
        pass

    def predict_single(self, image: np.ndarray) -> Dict[str, Any]:
        # TODO: preprocess -> forward -> postprocess for one image
        pass

    def predict_batch(self, images: np.ndarray) -> List[Dict[str, Any]]:
        # TODO: preprocess -> forward -> postprocess for batch
        pass

# TODO: Test on 10 random samples
# TODO: Print accuracy and average confidence
```

In [ ]:
# Try the exercise yourself before looking at the solution below!







### Solution

In [ ]:
# ----- Solution -----

class DigitInferencePipeline:
    """Self-contained inference pipeline for digit classification."""

    def __init__(self, checkpoint_path: str):
        ckpt = torch.load(checkpoint_path, weights_only=False)
        hp = ckpt["hyperparams"]
        self.model = SimpleMLP(
            input_dim=hp["input_dim"],
            hidden_dim=hp["hidden_dim"],
            output_dim=hp["output_dim"],
            dropout=hp.get("dropout", 0.0),
        )
        self.model.load_state_dict(ckpt["model_state_dict"])
        self.model.eval()
        self.metadata = {"epoch": ckpt["epoch"], "metrics": ckpt["metrics"]}
        print(f"Loaded digit classifier from epoch {ckpt['epoch']} "
              f"(acc={ckpt['metrics'].get('test_acc', 'N/A')})")

    def preprocess(self, images: np.ndarray) -> torch.Tensor:
        """Normalize [0,16] -> [0,1] and convert to float tensor."""
        images = images.astype(np.float32) / 16.0
        tensor = torch.from_numpy(images)
        if tensor.ndim == 1:
            tensor = tensor.unsqueeze(0)
        return tensor

    @torch.no_grad()
    def predict_single(self, image: np.ndarray) -> Dict[str, Any]:
        """Predict a single 8x8 image (can be 2D or flattened)."""
        flat = image.flatten()
        tensor = self.preprocess(flat)
        logits = self.model(tensor)
        probs = F.softmax(logits, dim=1).squeeze()
        pred = probs.argmax().item()
        return {
            "predicted_digit": pred,
            "confidence": probs[pred].item(),
            "all_probs": probs.tolist(),
        }

    @torch.no_grad()
    def predict_batch(self, images: np.ndarray) -> List[Dict[str, Any]]:
        """Predict a batch of flattened images."""
        tensor = self.preprocess(images)
        logits = self.model(tensor)
        probs = F.softmax(logits, dim=1)
        preds = probs.argmax(dim=1)

        results = []
        for i in range(len(images)):
            results.append({
                "predicted_digit": preds[i].item(),
                "confidence": probs[i, preds[i]].item(),
            })
        return results


# Test
pipe = DigitInferencePipeline(img_ckpt_path)

# Select 10 random test samples
set_global_seed(42)
test_indices = np.random.choice(len(X_img_test), size=10, replace=False)
test_images_raw = digits.data[perm[split:]][test_indices]  # Raw [0, 16]
test_labels = y_img_test[test_indices].numpy()

# Batch prediction
batch_preds = pipe.predict_batch(test_images_raw)

correct = 0
total_conf = 0.0
print(f"{'Sample':<8} {'True':<6} {'Predicted':<10} {'Confidence':<12} {'Correct'}")
print("-" * 50)
for i, (pred, true) in enumerate(zip(batch_preds, test_labels)):
    is_correct = pred["predicted_digit"] == true
    correct += int(is_correct)
    total_conf += pred["confidence"]
    mark = "yes" if is_correct else "NO"
    print(f"{i:<8} {true:<6} {pred['predicted_digit']:<10} "
          f"{pred['confidence']:<12.4f} {mark}")

print(f"\nAccuracy: {correct}/{len(test_labels)} = {correct/len(test_labels):.2%}")
print(f"Average confidence: {total_conf/len(test_labels):.4f}")

In [ ]:
# Clean up temp directory
shutil.rmtree(CHECKPOINT_DIR, ignore_errors=True)
print(f"Cleaned up {CHECKPOINT_DIR}")

---
## 11. Common Mistakes & Debugging Tips

**1. Forgetting `model.eval()` before inference**
- Dropout and BatchNorm behave differently in train vs eval mode.
- Always call `model.eval()` after loading for inference.

**2. Forgetting `torch.no_grad()` during inference**
- Without it, PyTorch builds a computation graph, wasting memory.
- Wrap all inference in `with torch.no_grad():` or use the `@torch.no_grad()` decorator.

**3. Saving the full model instead of `state_dict()`**
- `torch.save(model, path)` uses pickle and breaks when code changes.
- Always save `model.state_dict()` and reconstruct the architecture from hyperparameters.

**4. Architecture mismatch when loading**
- If the model class changed since saving, `load_state_dict` raises `RuntimeError`.
- Fix: save hyperparameters alongside weights so you can rebuild the exact architecture.
- Use `strict=False` only as a last resort (you may silently drop weights).

**5. Not saving the vocabulary for text models**
- The model is useless without the exact vocabulary used during training.
- Always save `vocab` inside the checkpoint or as a separate file.

**6. Preprocessing mismatch between training and inference**
- If training normalized to `[0, 1]` but inference sends raw pixels, predictions are wrong.
- Document and centralize preprocessing logic. Ideally, bundle it in the pipeline class.

**7. Device mismatch**
- Checkpoints saved on GPU may fail to load on CPU. Use `map_location`:
  ```python
  torch.load(path, map_location="cpu")
  ```

**8. Loading one sample at a time in production**
- Batch inference is 5-50x faster than single-sample loops.
- Collect inputs, stack into a batch, run one forward pass.

---

**Next notebook:** [03 - Performance Profiling & Batching](03_Performance_Profiling_and_Batching.ipynb) -- we cover batch size tradeoffs, gradient accumulation, DataLoader optimization, and profiling.